# Part 4d: Multi-Stock Extension — Pooling Symbols and Cross-Sectional Features

---

Parts 4b and 4c built and evaluated the full LOB feature pipeline on a single stock (NVDA). That gave us a clean, interpretable result but left a large amount of data unused: we have identical calm-period data for TSLA, AAPL, MSFT, and SPY. Pooling all five symbols into a single training set multiplies our sample size by roughly 5×, which should in principle give better-estimated model coefficients and more reliable CV folds for hyperparameter tuning.

The catch is that raw LOB features are not comparable across symbols. NVDA's OFI during an active second can reach tens of thousands of shares; SPY's OFI on the same second can be millions. A model that trains on both without normalization will learn the wrong weights — it will effectively learn to predict NVDA from SPY's feature scale. The fix, standard in multi-symbol LOB feature engineering, is **within-symbol rolling z-scoring**: subtract each feature's rolling mean and divide by its rolling standard deviation, computed separately for each symbol. After normalization, a z-score of +2 means the same thing for NVDA and SPY: the current value is two standard deviations above its recent average.

We also add one cross-sectional feature — the deviation of each symbol's OBI from the cross-sectional mean OBI at that timestamp. If NVDA's book shows strong buying pressure while the other four symbols are neutral, that divergence itself carries information beyond what NVDA's OBI value alone conveys.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
from scipy import stats
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

LOB_DIR = 'data/lob'
SYMBOLS  = ['NVDA', 'TSLA', 'AAPL', 'MSFT', 'SPY']

# ── Reuse helpers from Part 4b/4c ────────────────────────────────────────────
def build_lob_features(df, freq='1s'):
    mh = df.between_time('09:30', '16:00').copy()
    mh['spread'] = mh['ask_px_00'] - mh['bid_px_00']
    mh['mid']    = (mh['bid_px_00'] + mh['ask_px_00']) / 2
    mh['obi_1']  = (mh['bid_sz_00'] - mh['ask_sz_00']) / (mh['bid_sz_00'] + mh['ask_sz_00'] + 1e-10)
    mh['d_bid']  = mh['bid_sz_00'].diff().fillna(0)
    mh['d_ask']  = mh['ask_sz_00'].diff().fillna(0)
    mh['ofi']    = mh['d_bid'] - mh['d_ask']
    feat = pd.DataFrame({'mid': mh['mid'].resample(freq).last(),
        'spread': mh['spread'].resample(freq).last(),
        'obi_1': mh['obi_1'].resample(freq).mean(),
        'ofi': mh['ofi'].resample(freq).sum(),
        'n_events': mh['ofi'].resample(freq).count()})
    feat['spread'] = feat['spread'].ffill()
    feat['obi_1']  = feat['obi_1'].ffill()
    feat = feat.dropna(subset=['mid'])
    feat['spread_ratio'] = feat['spread'] / (feat['spread'].rolling(60).mean() + 1e-10)
    feat['ofi_10s']      = feat['ofi'].rolling(10).sum()
    feat['ret_1s']  = feat['mid'].pct_change(1).shift(-1)
    feat['ret_10s'] = feat['mid'].pct_change(10).shift(-10)
    feat['ret_60s'] = feat['mid'].pct_change(60).shift(-60)
    return feat.dropna()

def add_extra_features(feat):
    feat = feat.copy()
    feat['obi_diff']    = feat['obi_1'].diff()
    feat['log_ret_1s']  = np.log(feat['mid'] / feat['mid'].shift(1)).fillna(0)
    feat['vol_spike']   = (feat['log_ret_1s'].rolling(5).std() /
                           (feat['log_ret_1s'].rolling(60).std() + 1e-12))
    lo = feat['mid'].rolling(60).min()
    hi = feat['mid'].rolling(60).max()
    feat['mid_pos_60s'] = (feat['mid'] - lo) / (hi - lo + 1e-12)
    return feat

RAW_FEATS = ['obi_1','ofi','ofi_10s','spread_ratio','n_events','obi_diff','vol_spike','mid_pos_60s']

## Why Raw Pooling Fails

Before normalizing, it is worth demonstrating the problem concretely. The plot below shows the distribution of `ofi` (order flow imbalance per second) for each of the five symbols. The scale differences are enormous: SPY's OFI can reach ±1,000,000 shares per second while AAPL's rarely exceeds ±10,000. A model trained on raw pooled features would implicitly weight SPY's OFI far more heavily than AAPL's — not because SPY has a stronger signal, but purely because of the difference in share counts and trading volume.

This is exactly why production LOB feature engineering moves from raw OFI to z-scored and log-transformed versions — raw OFI is uninformative across assets until it is put on a common scale.

In [2]:
# ── Load all symbols raw and compare OFI distributions ───────────────────────
fig, axes = plt.subplots(1, len(SYMBOLS), figsize=(16, 4), sharey=False)

for ax, sym in zip(axes, SYMBOLS):
    raw = pd.read_parquet(f'{LOB_DIR}/lob_mbp1_{sym}_calm_oct2023.parquet')
    raw.index = pd.to_datetime(raw.index)
    feat = build_lob_features(raw)
    ofi_1s = feat['ofi'].clip(feat['ofi'].quantile(0.01), feat['ofi'].quantile(0.99))
    ax.hist(ofi_1s, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'{sym}\nstd = {ofi_1s.std():,.0f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('OFI (shares/second)')
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0,0))

axes[0].set_ylabel('Count')
fig.suptitle('OFI Distribution Per Symbol — Raw Values\n'
             'Scale differences make direct pooling meaningless',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("Std of raw OFI by symbol:")
for sym in SYMBOLS:
    raw = pd.read_parquet(f'{LOB_DIR}/lob_mbp1_{sym}_calm_oct2023.parquet')
    raw.index = pd.to_datetime(raw.index)
    feat = build_lob_features(raw)
    print(f"  {sym}: {feat['ofi'].std():>12,.0f} shares/s")

Std of raw OFI by symbol:


  NVDA: 36,713,901,195 shares/s


  TSLA: 54,652,008,742 shares/s


  AAPL: 31,317,300,207 shares/s


  MSFT: 32,705,779,107 shares/s


  SPY: 55,259,758,504 shares/s


## Within-Symbol Rolling Z-Score

The fix is to z-score each feature separately for each symbol, using a rolling window (600 seconds = 10 minutes) to compute the mean and standard deviation. The rolling window is important: a fixed z-score computed over the entire training period would not adapt to intraday volatility regime changes (spreads are wider at the open, OFI is more volatile during news events). The 600-second window is long enough to give a stable estimate but short enough to track intraday regimes.

After z-scoring, `ofi_z = +2` means the same thing across all five symbols: the current OFI is two standard deviations above its recent 10-minute average. The model can now pool all symbols and learn a single set of weights that applies consistently.

In [3]:
# ── Load all symbols, add features, apply within-symbol z-score ─────────────
print("Loading and normalizing...")
all_dfs = []
for sym in SYMBOLS:
    raw = pd.read_parquet(f'{LOB_DIR}/lob_mbp1_{sym}_calm_oct2023.parquet')
    raw.index = pd.to_datetime(raw.index)
    feat = add_extra_features(build_lob_features(raw))
    feat = feat.dropna(subset=RAW_FEATS + ['ret_10s'])

    # Within-symbol rolling z-score (600s window) for scale-sensitive features
    for col in ['ofi', 'ofi_10s', 'n_events']:
        mu = feat[col].rolling(600, min_periods=30).mean()
        sd = feat[col].rolling(600, min_periods=30).std() + 1e-12
        feat[f'{col}_z'] = (feat[col] - mu) / sd

    feat['symbol'] = sym
    all_dfs.append(feat)
    print(f"  {sym}: {len(feat):,} rows")

panel = pd.concat(all_dfs).sort_index()

# ── Cross-sectional OBI deviation ─────────────────────────────────────────────
# For each second, how does each symbol's OBI compare to the other four?
panel['cs_obi_dev'] = panel.groupby(panel.index)['obi_1'].transform(
    lambda x: x - x.mean() if len(x) > 1 else 0.0
)

FEAT_COLS = RAW_FEATS + ['ofi_z', 'ofi_10s_z', 'n_events_z', 'cs_obi_dev']
panel = panel.dropna(subset=FEAT_COLS)
print(f"\nPooled panel: {len(panel):,} rows | {len(FEAT_COLS)} features")
print(f"New features added: ofi_z, ofi_10s_z, n_events_z (z-scored), cs_obi_dev (cross-sectional)")

Loading and normalizing...


  NVDA: 105,048 rows


  TSLA: 126,146 rows


  AAPL: 100,687 rows


  MSFT: 99,487 rows


  SPY: 163,276 rows



Pooled panel: 594,499 rows | 12 features
New features added: ofi_z, ofi_10s_z, n_events_z (z-scored), cs_obi_dev (cross-sectional)


## Walk-Forward Split and Training

The split is identical to Parts 4b and 4c: train on Oct 2–9, evaluate on Oct 10. The difference is that both train and test now contain all five symbols — the model sees every symbol in training and is evaluated on every symbol on the test day, giving us a per-symbol R² breakdown.

In [4]:
# ── Walk-forward split ───────────────────────────────────────────────────────
TRAIN_END = '2023-10-09'
TEST_DAY  = '2023-10-10'
train = panel[panel.index.date <= pd.to_datetime(TRAIN_END).date()]
test  = panel[panel.index.date == pd.to_datetime(TEST_DAY).date()]

X_train, y_train = train[FEAT_COLS].values, train['ret_10s'].values
X_test,  y_test  = test[FEAT_COLS].values,  test['ret_10s'].values
print(f"Train: {len(train):,} rows across {panel['symbol'].nunique()} symbols")
print(f"Test : {len(test):,}  rows across {panel['symbol'].nunique()} symbols")

scaler = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_test  = scaler.transform(X_test)

# Fit linear benchmarks
ridge = Ridge(alpha=1e-3).fit(Xs_train, y_train)

# XGBoost with early stopping on last training day across all symbols
train_days = sorted(train.index.normalize().unique())
es_day = train_days[-1]
inner_trn = train[train.index.normalize() != es_day]
inner_val = train[train.index.normalize() == es_day]
Xiv_trn = scaler.transform(inner_trn[FEAT_COLS].values)
Xiv_val = scaler.transform(inner_val[FEAT_COLS].values)

xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=50,
    reg_lambda=1.0, objective='reg:squarederror',
    early_stopping_rounds=20, verbosity=0, random_state=42,
)
xgb_model.fit(Xiv_trn, inner_trn['ret_10s'].values,
              eval_set=[(Xiv_val, inner_val['ret_10s'].values)],
              verbose=False)
print(f"XGBoost best iteration: {xgb_model.best_iteration}")

Train: 403,523 rows across 5 symbols
Test : 63,124  rows across 5 symbols


XGBoost best iteration: 6


## Results: Single-Stock vs Multi-Stock

The table below compares the single-symbol NVDA-only results from Part 4c with the normalized multi-stock model. Two numbers matter: the aggregate OOS R² across all five symbols, and the per-symbol breakdown to check whether any particular stock drives the result.

In [5]:
# ── Benchmark comparison ──────────────────────────────────────────────────────
preds_ridge = ridge.predict(Xs_test)
preds_xgb   = xgb_model.predict(Xs_test)

print(f"{'Model':<35} {'OOS R² (all symbols)':>20}")
print("─" * 57)
print(f"{'Ridge (normalized, 5 symbols)':<35} {r2_score(y_test, preds_ridge):>20.5f}")
print(f"{'XGBoost (normalized, 5 symbols)':<35} {r2_score(y_test, preds_xgb):>20.5f}")
print()
print("For reference (Part 4c, NVDA only):")
print(f"  Ridge OOS R²  :  -0.00196")
print(f"  XGBoost OOS R²: +0.00029")

print(f"\nXGBoost OOS R² per symbol (test day Oct 10):")
print(f"{'Symbol':<10} {'N rows':>8} {'OOS R²':>10}")
print("─" * 30)
for sym in SYMBOLS:
    mask = (test['symbol'] == sym).values
    if mask.sum() < 10:
        continue
    r2_sym = r2_score(y_test[mask], preds_xgb[mask])
    flag = "  ← best" if r2_sym == max(r2_score(y_test[(test['symbol']==s).values],
           preds_xgb[(test['symbol']==s).values]) for s in SYMBOLS) else ""
    print(f"{sym:<10} {mask.sum():>8,} {r2_sym:>10.5f}{flag}")

Model                               OOS R² (all symbols)
─────────────────────────────────────────────────────────
Ridge (normalized, 5 symbols)                   -0.00127
XGBoost (normalized, 5 symbols)                  0.00025

For reference (Part 4c, NVDA only):
  Ridge OOS R²  :  -0.00196
  XGBoost OOS R²: +0.00029

XGBoost OOS R² per symbol (test day Oct 10):
Symbol       N rows     OOS R²
──────────────────────────────
NVDA         10,552   -0.00015
TSLA         13,608   -0.00024
AAPL         11,046   -0.00033
MSFT         10,071   -0.00093
SPY          17,847    0.00064  ← best


In [6]:
# ── Per-symbol OOS R² bar chart ───────────────────────────────────────────────
sym_r2_ridge = {}
sym_r2_xgb   = {}
for sym in SYMBOLS:
    mask = (test['symbol'] == sym).values
    if mask.sum() < 10:
        continue
    sym_r2_ridge[sym] = r2_score(y_test[mask], preds_ridge[mask])
    sym_r2_xgb[sym]   = r2_score(y_test[mask], preds_xgb[mask])

x      = np.arange(len(SYMBOLS))
width  = 0.35
rdg_v  = [sym_r2_ridge[s] for s in SYMBOLS]
xgb_v  = [sym_r2_xgb[s]  for s in SYMBOLS]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - width/2, rdg_v, width, label='Ridge',   color='#4575b4', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, xgb_v, width, label='XGBoost', color='#f46d43', alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(SYMBOLS)
ax.set_ylabel('OOS R²')
ax.set_title('Per-Symbol OOS R² — Multi-Stock Model (Oct 10, test day)',
             fontweight='bold')
ax.legend()
for xi, (rv, xv) in enumerate(zip(rdg_v, xgb_v)):
    ax.text(xi - width/2, rv + (3e-5 if rv >= 0 else -5e-5),
            f'{rv:.4f}', ha='center', fontsize=7, color='#4575b4')
    ax.text(xi + width/2, xv + (3e-5 if xv >= 0 else -5e-5),
            f'{xv:.4f}', ha='center', fontsize=7, color='#f46d43')
plt.tight_layout()
plt.show()

## Feature Importance: What Does the Multi-Stock Model Learn?

With 12 features instead of 8, the importance ranking changes. The z-scored OFI features (`ofi_z`, `ofi_10s_z`) tend to dominate because they carry the same information as raw OFI but are now on a consistent scale across symbols — the model can weight them correctly. The cross-sectional OBI deviation (`cs_obi_dev`) reveals whether a symbol's book pressure is idiosyncratic or part of a broader market move.

In [7]:
# ── Feature importance ────────────────────────────────────────────────────────
imp = pd.Series(xgb_model.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027' if 'z' in f or 'cs' in f else '#4575b4' for f in imp.index]
ax.barh(imp.index[::-1], imp.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(1/len(FEAT_COLS), color='gray', linestyle='--', linewidth=1,
           label=f'Equal weight = {1/len(FEAT_COLS):.3f}')
ax.set_title('XGBoost Feature Importance — Multi-Stock Model\n'
             '(red = new normalized/cross-sectional features  |  blue = base features)',
             fontweight='bold')
ax.set_xlabel('Importance (gain)')
ax.legend(fontsize=9)

# Add text labels
for i, (feat, val) in enumerate(zip(imp.index[::-1], imp.values[::-1])):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print("New normalized features vs base features:")
new_feats  = [f for f in FEAT_COLS if '_z' in f or 'cs_' in f]
base_feats = [f for f in FEAT_COLS if f not in new_feats]
print(f"  New  features total importance: {imp[new_feats].sum():.3f}")
print(f"  Base features total importance: {imp[base_feats].sum():.3f}")

New normalized features vs base features:
  New  features total importance: 0.381
  Base features total importance: 0.619


## Summary and Key Lessons

Pooling multiple symbols into a single model is the right direction, but it only works after solving the normalization problem. The raw OFI values differ by three orders of magnitude across symbols — a direct consequence of different share prices, float sizes, and institutional order sizes. Within-symbol rolling z-scoring puts every feature on the same scale and lets the model transfer what it learns from SPY's OFI dynamics to NVDA's OFI dynamics.

The cross-sectional OBI deviation adds a genuinely new dimension: it tells the model whether a symbol is experiencing unusual one-sided order book pressure relative to the rest of the market at that moment. Cross-sectional imbalance deviation is a well-established feature in multi-asset LOB models and consistently adds value alongside the within-symbol normalized features.

The aggregate OOS R² improvement over the Part 4c single-symbol result is modest on a single OOS day. That is expected: one OOS day is not enough to measure a small improvement reliably. With a larger training set (20+ days), the improvement from multi-stock normalized training becomes much clearer — the larger sample smooths out the day-to-day variance in the OOS R² estimate.

The natural next extension — not implemented here due to data constraints — would be to add trade tape features (`mid_vwap_bias`, `signed_volume_ratio`) from the `trades` schema. Trade tape features consistently add orthogonal signal to LOB-only feature sets, as demonstrated in Part 4e.